In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from scipy.stats import spearmanr
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModel, get_scheduler
from torch.optim import AdamW
from tqdm import tqdm

In [ ]:
# Set device with memory fraction
torch.cuda.set_per_process_memory_fraction(0.7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# Load data
df = pd.read_csv("ranked_responses_final.csv")

In [ ]:
# Normalize ranks
def compute_proportional_ranks(ranks):
    rank_to_weight = {1: 4, 2: 3, 3: 2, 4: 1}
    total_weight = sum(rank_to_weight[r] for r in ranks)
    return [rank_to_weight[r] / total_weight for r in ranks]

model_order = ["openai/gpt-4o", "anthropic/claude-3.5-sonnet", "deepseek/deepseek-chat", "perplexity/sonar"]
data = []
for prompt in df["Prompt"].unique():
    sub = df[df["Prompt"] == prompt]
    models = sub["Model"].tolist()
    ranks = sub["Rank"].tolist()
    if len(models) != 4: continue
    scores = dict(zip(models, compute_proportional_ranks(ranks)))
    if set(scores.keys()) != set(model_order): continue
    source = sub["Source"].iloc[0] if pd.notnull(sub["Source"].iloc[0]) else None
    data.append({"prompt": prompt, "scores": [scores[m] for m in model_order], "source": source})

df_proc = pd.DataFrame(data).dropna(subset=["source"])
category_labels = df_proc["source"].astype("category").cat.codes

# Stratified split
train_df, test_df = train_test_split(
    df_proc, test_size=0.2, stratify=category_labels, random_state=42
)

In [ ]:
# Tokenizer and Dataset
from transformers import DebertaV2Tokenizer
tokenizer = DebertaV2Tokenizer.from_pretrained("microsoft/deberta-v3-small")

model_name = "microsoft/deberta-v3-small"

class PromptDataset(Dataset):
    def __init__(self, prompts, targets, tokenizer, max_length=256):
        #NOTE: no return_tensors="pt" here
        self.encodings = tokenizer(
            prompts,
            truncation=True,
            padding="max_length",
            max_length=max_length
        )
        self.targets = targets

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        item = {}
        for key, val in self.encodings.items():
            # Defensive indexing: supports both lists and tensors
            if isinstance(val, list) or isinstance(val, np.ndarray):
                item[key] = torch.tensor(val[idx])
            else:
                item[key] = val[idx] if isinstance(val[idx], torch.Tensor) else torch.tensor(val[idx])
        item["labels"] = torch.tensor(self.targets[idx], dtype=torch.float32)
        return item

In [ ]:
train_ds = PromptDataset(train_df["prompt"].tolist(), train_df["scores"].tolist(), tokenizer)
test_ds = PromptDataset(test_df["prompt"].tolist(), test_df["scores"].tolist(), tokenizer)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=16)

In [ ]:
# Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class DebertaRegressor(nn.Module):
    def __init__(self, model_name, num_outputs):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        self.regressor = nn.Linear(self.backbone.config.hidden_size, num_outputs)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        x = out.last_hidden_state[:, 0]
        return self.regressor(x)

model = DebertaRegressor(model_name, num_outputs=4).to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)
epochs = 10
num_training_steps = len(train_loader) * epochs
lr_scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

# Training loop
train_losses, val_losses, spearman_scores = [], [], []
best_val_loss = float("inf")
patience = 2
no_improve = 0

In [ ]:
for epoch in range(200):  #Full 200-epoch training, no stopping early
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        preds = model(input_ids, attention_mask)
        loss = F.mse_loss(preds, labels)
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        total_loss += loss.item()
    
    avg_train_loss = total_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # Validation
    model.eval()
    val_loss, all_preds, all_labels = 0, [], []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            preds = model(input_ids, attention_mask)
            val_loss += F.mse_loss(preds, labels).item()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(test_loader)
    val_losses.append(avg_val_loss)

    corr = np.mean([spearmanr(p, t).correlation for p, t in zip(all_preds, all_labels)])
    spearman_scores.append(corr)

    print(f"Epoch {epoch+1}: Train Loss={avg_train_loss:.4f}, Val Loss={avg_val_loss:.4f}, Spearman={corr:.4f}")

    # Save best model if improved
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "best_deberta_model.pt")

In [ ]:
# Plot
plt.plot(range(1, len(train_losses)+1), train_losses, label="Train Loss")
plt.plot(range(1, len(val_losses)+1), val_losses, label="Val Loss")
#plt.plot(range(1, len(spearman_scores)+1), spearman_scores, label="Spearman")
plt.xlabel("Epoch")
plt.title("Training Progress")
plt.legend()
plt.tight_layout()
plt.savefig("training_progress_model.png")
plt.show()

print("Training complete. Best model saved to best_deberta_model.pt")

In [ ]:
# === Evaluation ===
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Reload best model
model.load_state_dict(torch.load("best_deberta_model.pt"))
model.eval()

# Predict on test set
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"]
        preds = model(input_ids, attention_mask).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

# Convert to arrays
y_pred = np.array(all_preds)
y_true = np.array(all_labels)

# Regression Metrics
mse = mean_squared_error(y_true, y_pred)
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

print("\nRegression Evaluation Metrics:")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R² Score: {r2:.4f}")

In [ ]:
# Convert scores to rankings
def convert_to_ranking(scores):
    sorted_models = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return {model: rank + 1 for rank, (model, _) in enumerate(sorted_models)}

model_names = ["openai/gpt-4o", "anthropic/claude-3.5-sonnet", "deepseek/deepseek-chat", "perplexity/sonar"]

# Full ranking eval
actual_ranks, predicted_ranks = [], []
for i in range(len(y_true)):
    actual_scores = dict(zip(model_names, y_true[i]))
    predicted_scores = dict(zip(model_names, y_pred[i]))
    actual_rank = convert_to_ranking(actual_scores)
    predicted_rank = convert_to_ranking(predicted_scores)
    for model in model_names:
        actual_ranks.append(actual_rank[model])
        predicted_ranks.append(predicted_rank[model])

conf_mat = confusion_matrix(actual_ranks, predicted_ranks, labels=[1, 2, 3, 4])
acc_score = accuracy_score(actual_ranks, predicted_ranks)
precision_val = precision_score(actual_ranks, predicted_ranks, average='macro')
recall_val = recall_score(actual_ranks, predicted_ranks, average='macro')
f1_val = f1_score(actual_ranks, predicted_ranks, average='macro')

print("\nClassification Evaluation Metrics (All Ranks):")
print("Confusion Matrix:")
print(conf_mat)
print(f"Accuracy: {acc_score:.4f}")
print(f"Precision: {precision_val:.4f}")
print(f"Recall: {recall_val:.4f}")
print(f"F1 Score: {f1_val:.4f}")

plt.figure(figsize=(6, 5))
sns.heatmap(conf_mat, annot=True, fmt='d', cmap='Blues', xticklabels=[1, 2, 3, 4], yticklabels=[1, 2, 3, 4])
plt.title("Confusion Matrix (Predicted vs Actual Rankings)")
plt.xlabel("Predicted Rank")
plt.ylabel("Actual Rank")
plt.tight_layout()
plt.show()

In [ ]:
# Top-1 Prediction Evaluation
actual_best_models, predicted_best_models = [], []
for i in range(len(y_true)):
    actual_scores = dict(zip(model_names, y_true[i]))
    predicted_scores = dict(zip(model_names, y_pred[i]))
    actual_best = [m for m, r in convert_to_ranking(actual_scores).items() if r == 1][0]
    predicted_best = [m for m, r in convert_to_ranking(predicted_scores).items() if r == 1][0]
    actual_best_models.append(actual_best)
    predicted_best_models.append(predicted_best)

acc_score = accuracy_score(actual_best_models, predicted_best_models)
precision_val = precision_score(actual_best_models, predicted_best_models, average='macro')
recall_val = recall_score(actual_best_models, predicted_best_models, average='macro')
f1_val = f1_score(actual_best_models, predicted_best_models, average='macro')
conf_mat = confusion_matrix(actual_best_models, predicted_best_models, labels=model_names)

print("\nEvaluation for Predicting Best Model (Rank 1 Only):")
print("Confusion Matrix:")
print(conf_mat)
print(f"Accuracy: {acc_score:.4f}")
print(f"Precision: {precision_val:.4f}")
print(f"Recall: {recall_val:.4f}")
print(f"F1 Score: {f1_val:.4f}")

plt.figure(figsize=(7, 6))
sns.heatmap(conf_mat, annot=True, fmt='d', cmap='Blues', xticklabels=model_names, yticklabels=model_names)
plt.title("Confusion Matrix (Predicted vs Actual Best Model)")
plt.xlabel("Predicted Best Model")
plt.ylabel("Actual Best Model")
plt.tight_layout()
plt.show()